# Lab 3b — Extract Entities and Answer Questions | استخراج الكيانات والإجابة على الأسئلة
**SDA-AIE-211 | Day2 **

## القصة | The story
- 🇸🇦 صنّف النموذج الشكوى ووجّهها لفريق الخدمة. الآن يجب على بيان **استخراج الحقول المهمة** (الشخص، المكان، الخدمة) و**الإجابة على أسئلة** من وثائق الخدمة، وأن يقول **"لا توجد إجابة"** عندما لا تكون الإجابة موجودة.
- 🇬🇧 The classifier routed the complaint. Now Bayan must **extract key fields** (person, location, service) and **answer questions** from service documents — and say **"no answer"** when the answer isn't there.

## الخطوات | Steps
| # | الخطوة | Step |
|---|---|---|
| 1 | تحميل بيانات الكيانات | Load NER data |
| 2 | ربط الوسوم بقطع الكلمات | Align labels to subwords |
| 3 | تدريب وقياس NER | Train and score NER |
| 4 | الأسئلة والإجابات (QA) | Question answering |
| 5 | تدريب نموذج QA الخاص ببيان | Train our own QA model |
| 6 | اختبار نموذج QA | Test the QA model |

## Step 1 — تحميل البيانات | Load data

In [2]:
!pip -q install pandas numpy scikit-learn torch transformers datasets accelerate seqeval

from pathlib import Path
import zipfile, json, re
import numpy as np
import pandas as pd
import torch


DATA_DIR = Path('/content/')           # مجلد البيانات | data folder
ARTIFACT_DIR = Path('/content/artifacts')  # مجلد النتائج | results folder
ARTIFACT_DIR.mkdir(exist_ok=True)

ner = pd.read_csv(DATA_DIR / 'bayan_ner_4000.csv')
ner['words'] = ner.tokens_json.map(json.loads)       # الكلمات | words
ner['tags'] = ner.bio_tags_json.map(json.loads)      # الوسوم | tags
print('Rows:', len(ner))
print(ner[['text', 'tags']].head(3).to_string())

# B- = بداية الكيان | beginning · I- = استمرار | inside · O = ليس كيانًا | outside
tag_names = sorted({t for row in ner.tags for t in row})
tag_to_id = {t: i for i, t in enumerate(tag_names)} #add ID to each tag
print('Tags:', tag_names)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Rows: 4000
                                                                        text                                                                                  tags
0                             هل مشكلة خدمة الحافلات في النرجس مرتبطة بـ جدة                         [O, O, B-SERVICE, I-SERVICE, O, B-LOCATION, O, O, B-LOCATION]
1                     Is the billing problem in Riyadh related to Al Nakheel                     [O, O, B-SERVICE, O, O, B-LOCATION, O, O, B-LOCATION, I-LOCATION]
2  Please escalate the permit application case of Amal Al-Shehri in Al Malqa  [O, O, O, B-SERVICE, I-SERVICE, O, O, B-PERSON, I-PERSON, O, B-LOCATION, I-LOCATION]
Tags: ['B-LOCATION', 'B-PERSON', 'B-SERVICE', 'I-LOCATION', 'I-PERSON', 'I-SERVICE', 'O']


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
tag_to_id

{'B-LOCATION': 0,
 'B-PERSON': 1,
 'B-SERVICE': 2,
 'I-LOCATION': 3,
 'I-PERSON': 4,
 'I-SERVICE': 5,
 'O': 6}

## Step 2 — ربط الوسوم بقطع الكلمات | Align labels to subwords
- 🇸🇦 النموذج يقطّع الكلمة إلى أجزاء (subwords). نعطي **أول جزء** وسم الكلمة، والباقي `-100` (يعني: تجاهل).
- 🇬🇧 The model splits a word into subwords. The **first piece** gets the word's tag, the rest get `-100` (ignore).
- 🔒 الدالة جاهزة / the function is read-only.

In [5]:
from transformers import AutoTokenizer
CHECKPOINT = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

# Define a function to align word-level labels with subword tokens
def align_labels(words, tags):

    # Tokenize the input words and allow each word to be split into subwords
    # Truncate the sequence if it exceeds 128 tokens
    enc = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    # Create an empty list to store the aligned labels
    # Initialize previous to None to track the previous word ID
    labels, previous = [], None

    # Loop through the word ID of each token produced by the tokenizer
    for word_id in enc.word_ids():

        # If word_id is None, the token is a special token such as [CLS] or [SEP]
        # Assign -100 so the model ignores it during training
        if word_id is None:
            labels.append(-100)

        # If the word ID is different from the previous one,
        # this is the first subword of a new word
        # Assign the original label of that word
        elif word_id != previous:
            labels.append(tag_to_id[tags[word_id]])

        # Otherwise, this is a continuation subword of the same word
        # Assign -100 so it is ignored during training
        else:
            labels.append(-100)

        # Update previous with the current word ID
        # This helps identify the first and continuation subwords
        previous = word_id

    # Add the aligned labels to the tokenizer output
    enc['labels'] = labels

    # Return tokenized inputs together with their aligned labels
    return enc


# Select the first example from the NER dataset
row = 0

# Apply the label alignment function to the selected example
# ner.words contains the original words
# ner.tags contains the corresponding word-level labels
example = align_labels(ner.words.iloc[row], ner.tags.iloc[row])

# Convert token IDs back into readable token strings and display them
print(tokenizer.convert_ids_to_tokens(example['input_ids']))

# Display the aligned numerical labels
print(example['labels']) #final output tokens and their labels(tags to ID)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

['<s>', '▁هل', '▁مشكلة', '▁خدمة', '▁ال', 'حاف', 'لات', '▁في', '▁الن', 'رج', 'س', '▁مرتبط', 'ة', '▁ب', 'ـ', '▁جد', 'ة', '</s>']
[-100, 6, 6, 2, 5, -100, -100, 6, 0, -100, -100, 6, -100, 6, -100, 0, -100, -100]


### Steps: Word-Level Label Alignment with Subword Tokens

1. **Tokenization:** The tokenizer splits each original word into one or more Subword Tokens and assigns a unique Token ID to each token.

2. **Word IDs:** Each token is assigned a Word ID that identifies which original word it belongs to. Subwords from the same word share the same Word ID.

3. **Label Alignment:** Use the Word ID to retrieve the original NER label from the `tags` list.

4. **First Subword:** Assign the original word's Label ID to the first subword.

5. **Continuation Subwords:** Assign `-100` to the remaining subwords of the same word so they are ignored during loss calculation.

6. **Special Tokens:** Assign `-100` to special tokens such as `<s>` and `</s>` because they do not represent original words and should be ignored during loss calculation.

**Final Result:** The tokenizer output contains `input_ids`, `attention_mask`, and aligned `labels`, all in the same sequence order, ready for NER model training.

## Step 3 — تدريب وقياس NER | Train and score NER
- 🇸🇦 نقيس على **مستوى الكيان** (Entity F1) وليس على مستوى الجزء. الكيان صحيح فقط إذا طابق **كل** أجزائه.
- 🇬🇧 We score at **entity level** (Entity F1), not per subword. An entity counts only if **all** of it is right.
- نحن لا نريد فقط معرفة هل النموذج صنّف كل Token بشكل صحيح، بل نريد معرفة هل استطاع اكتشاف الكيان كاملًا وبحدوده الصحيحة.

### Steps: NER Model Training and Evaluation

1. **Import Libraries:** Import the libraries required for dataset preparation, model training, and evaluation.

2. **Split Dataset:** Split the data into 85% Training and 15% Validation sets.

3. **Encode Batch:** Apply Tokenization and Label Alignment to convert words and labels into model inputs.

4. **Make Dataset:** Convert the encoded data into Hugging Face Dataset format.

5. **Load Pretrained Model:** Load the pretrained Transformer model and add a Token Classification Head for NER.

6. **Define Metrics:** Define Precision, Recall, and Entity-level F1 to evaluate model performance.

7. **Configure Training:** Set the Learning Rate, Batch Size, Number of Epochs, and other training parameters.

8. **Train Model:** Fine-tune the pretrained model using the Training Dataset.

9. **Evaluate Model:** Evaluate the trained model using the Validation Dataset and calculate performance metrics.

10. **Save Model:** Save the fine-tuned model and tokenizer for future use.

In [6]:
# Import Hugging Face Dataset to convert the data into a format
# that can be used by the Transformers Trainer
from datasets import Dataset

# Import the model, training configuration, Trainer, and data collator
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# Import a function to split the dataset into training and validation sets
from sklearn.model_selection import train_test_split

# Import evaluation metrics for Named Entity Recognition (NER)
from seqeval.metrics import (
    f1_score as entity_f1,
    precision_score,
    recall_score
)


# Split the NER dataset into training (85%) and validation (15%) sets
# random_state=42 ensures the same split is produced each time
train_rows, valid_rows = train_test_split(
    ner,
    test_size=0.15,
    random_state=42
)


# Define a function to tokenize a batch of examples and align their labels
def encode_batch(batch):

    # Apply the align_labels function to each example
    # zip() pairs the words with their corresponding NER tags
    rows = [
        align_labels(w, t)
        for w, t in zip(batch['words'], batch['tags'])
    ]

    # Collect input_ids, attention_mask, and labels from each encoded example
    # Return them as lists in a dictionary
    return {
        k: [r[k] for r in rows]
        for k in ['input_ids', 'attention_mask', 'labels']
    }


# Define a function to convert a DataFrame into a Hugging Face Dataset
def make_ds(frame):

    # Create a Dataset containing the original words and their NER tags
    ds = Dataset.from_dict({
        'words': frame.words.tolist(),
        'tags': frame.tags.tolist()
    })

    # Apply encode_batch to tokenize the data and align the labels
    # batched=True processes multiple examples together
    # remove_columns removes the original words and tags after encoding
    return ds.map(
        encode_batch,
        batched=True,
        remove_columns=['words', 'tags']
    )


# Convert the training and validation DataFrames into encoded datasets
train_ds, valid_ds = make_ds(train_rows), make_ds(valid_rows)


# Load a pretrained Transformer model for token classification (NER)
model = AutoModelForTokenClassification.from_pretrained(

    CHECKPOINT,  # Pretrained model checkpoint

    num_labels=len(tag_names),  # Number of NER labels the model must predict

    id2label={i: t for i, t in enumerate(tag_names)},  # Map label IDs to label names

    label2id=tag_to_id  # Map label names to their corresponding numerical IDs
)


# Define a function to calculate NER evaluation metrics
def ner_metrics(eval_pred):

    # Extract the model's prediction scores (logits) and the true labels
    logits, gold = eval_pred

    # Select the predicted label with the highest score for each token
    # axis=2 selects the highest-scoring label across all label classes
    pred_ids = np.argmax(logits, axis=2)

    # Create lists to store the true and predicted entity tags
    true_tags, pred_tags = [], []

    # Process each example in the batch
    for p_row, y_row in zip(pred_ids, gold):

        # Keep only tokens whose true label is not -100
        # Convert numerical label IDs into their original tag names
        keep = [
            (tag_names[int(y)], tag_names[int(p)])
            for p, y in zip(p_row, y_row)
            if y != -100
        ]

        # Store the true labels for this example
        true_tags.append([a for a, _ in keep])

        # Store the predicted labels for this example
        pred_tags.append([b for _, b in keep])

    # Calculate entity-level F1, Precision, and Recall using seqeval
    return {
        'entity_f1': entity_f1(true_tags, pred_tags),
        'entity_precision': precision_score(true_tags, pred_tags),
        'entity_recall': recall_score(true_tags, pred_tags)
    }


# Configure the model training and evaluation settings
args = TrainingArguments(

    output_dir='/content/bayan_ner_model',  # Directory to save the model

    learning_rate=2e-5,  # Small learning rate for fine-tuning the pretrained model

    per_device_train_batch_size=16,  # Number of training examples processed per training step

    per_device_eval_batch_size=16,  # Number of validation examples processed at a time

    num_train_epochs=1,  # Number of complete passes through the training dataset

    eval_strategy='epoch',  # Evaluate the model at the end of each epoch

    save_strategy='no',  # Do not save intermediate checkpoints during training

    fp16=torch.cuda.is_available(),  # Enable mixed-precision training if a CUDA GPU is available

    report_to='none'  # Disable external experiment tracking integrations
)


# Create the Trainer to manage the NER training and evaluation process
trainer = Trainer(

    model=model,  # Pretrained Transformer model with a token classification head

    args=args,  # Training configuration

    train_dataset=train_ds,  # Encoded dataset used for training

    eval_dataset=valid_ds,  # Encoded dataset used for validation

    data_collator=DataCollatorForTokenClassification(tokenizer),  # Dynamically pad inputs and labels

    compute_metrics=ner_metrics  # Calculate entity-level evaluation metrics
)


# Start fine-tuning the pretrained NER model
trainer.train()


# Evaluate the trained model on the validation dataset
result = trainer.evaluate()


# Save the fine-tuned model and its configuration
trainer.save_model('/content/bayan_ner_model')

# Save the tokenizer so the same tokenization can be used later
tokenizer.save_pretrained('/content/bayan_ner_model')


# Extract the entity-level F1 score from the evaluation results
entity_score = result['eval_entity_f1']

# Display the final entity-level F1 score with three decimal places
print(f'Entity-level F1 / درجة الكيان: {entity_score:.3f}')

Map:   0%|          | 0/3400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Entity F1,Entity Precision,Entity Recall
1,No log,0.001954,1.000000,1.000000,1.000000


Training Loss,Validation Loss,Epoch,Entity F1,Entity Precision,Entity Recall
No log,0.001954,1,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Entity-level F1 / درجة الكيان: 1.000


In [7]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ner_path = Path("/content/drive/MyDrive/bayan_ner_model")
trainer.save_model(str(ner_path))
tokenizer.save_pretrained(str(ner_path))

assert (ner_path / "config.json").is_file()
print("NER saved:", ner_path)
print("Validation:", trainer.evaluate())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NER saved: /content/drive/MyDrive/bayan_ner_model


Training Loss,Validation Loss,Epoch,Entity F1,Entity Precision,Entity Recall
No log,0.001954,1,1.000000,1.000000,1.000000


Validation: {'eval_loss': 0.0019535047467797995, 'eval_entity_f1': 1.0, 'eval_entity_precision': 1.0, 'eval_entity_recall': 1.0}


## Step 4 — الأسئلة والإجابات | Question answering
- 🇸🇦 النموذج يبحث عن **مقطع** داخل النص يجيب السؤال. إن كانت درجة "لا إجابة" أعلى من أفضل مقطع نُرجع `None` بدل إجابة مختلقة.
- 🇬🇧 The model finds a **span** inside the text. If the "no answer" score beats the best span we return `None` instead of inventing an answer.

In [10]:
qa_rows = json.load(open(DATA_DIR / 'qa_smoke_set.json', encoding='utf-8'))

# Display the question from the first example and check whether it is unanswerable.
# is_impossible = True means the answer does not exist in the provided context.
print(qa_rows[0]['question'], '|', 'impossible =', qa_rows[0]['is_impossible']) #is there an answer in the doc

# Count how many questions in the dataset are unanswerable.
n_no_answer = sum(q['is_impossible'] for q in qa_rows)

# Display the total number of questions, answerable questions, and unanswerable questions.
print(f'Total: {len(qa_rows)} | answerable: {len(qa_rows) - n_no_answer} | NO answer: {n_no_answer}')

What is the refund processing fee? | impossible = True
Total: 12 | answerable: 9 | NO answer: 3


### Steps: Finding the Best Answer Span

1. **Get Start and End Logits**
   - The model predicts scores for each token as a possible answer start or end.

2. **Select Top 20 Positions**
   - Choose the 20 highest-scoring start positions and the 20 highest-scoring end positions.

3. **Generate Valid Pairs**
   - Combine start and end positions and remove invalid pairs.

4. **Calculate Span Scores**
   - Calculate a score for each valid span and select the highest-scoring one.

5. **Compare with Null Score**
   - Compare the best span score with the null score to decide whether to return an answer or `None`.

6. **Apply Offset Mapping**
   - Use offsets to extract the selected answer from its exact position in the original context.

7. **Compare with Ground Truth**
   - During evaluation, compare the predicted answer with the correct answer to measure how accurate the model is.

In [11]:
#بحث عن أفضل جزء من النص يمكن أن يكون إجابة للسؤال
def best_span(start_logits, end_logits, offsets,
              context, null_threshold=0.0, max_len=48, top_k=20):

    # Calculate the score for the null answer (no answer in the context).
    # Index 0 represents the [CLS] token.
    null_score = float(start_logits[0] + end_logits[0])

    # Initialize the best answer with a very low score.
    best = {'score': -1e9, 'text': ''}

    # Select the top-k token positions with the highest start scores.
    for s in np.argsort(start_logits)[-top_k:]:

        # Select the top-k token positions with the highest end scores.
        for e in np.argsort(end_logits)[-top_k:]:

            # Skip invalid spans:
            # - Do not allow [CLS] as an answer.
            # - The end position must not come before the start.
            # - The answer must not exceed the maximum allowed length.
            if s == 0 or e == 0 or e < s or e - s + 1 > max_len:
                continue

            # Skip tokens that do not map to the original context text.
            # This excludes question tokens and special tokens.
            if offsets[s] is None or offsets[e] is None:
                continue

            # Calculate the answer span score using start and end logits.
            score = float(start_logits[s] + end_logits[e])

            # Keep the candidate with the highest score.
            if score > best['score']:
                best = {
                    'score': score,

                    # Convert token positions into the original text
                    # using character offsets.
                    'text': context[offsets[s][0]:offsets[e][1]]
                }

    # Compare the null score with the best possible answer score.
    # If the null score is higher by more than the threshold,
    # the model decides that no answer exists in the context.
    if null_score - best['score'] > null_threshold:
        return {
            'answer': None,
            'reason': 'no_answer_in_context'
        }

    # Otherwise, return the best answer found in the context.
    return {
        'answer': best['text'],
        'score': best['score']
    }

## Step 5 — تدريب نموذج QA الخاص ببيان | Train our own QA model
- 🇸🇦 لا يوجد نموذج QA رسمي في ملفات الدورة، لذلك **ندرّب نموذجنا** على `bayan_qa_600.jsonl` ونحفظه باسم `bayan_qa_model`.
- 🇬🇧 The course names no official QA model, so we **train our own** on `bayan_qa_600.jsonl` and save it as `bayan_qa_model`.
- 🇸🇦 الفكرة: نعلّم النموذج أن يضع علامة **بداية** وعلامة **نهاية** للإجابة داخل النص. إذا لم توجد إجابة، تشير العلامتان إلى أول رمز (رقم 0).
- 🇬🇧 Idea: teach the model to mark the **start** and **end** of the answer inside the text. If there is no answer, both marks point to the first token (number 0).

In [13]:
from datasets import Dataset
from transformers import AutoModelForQuestionAnswering, DataCollatorWithPadding
from sklearn.model_selection import GroupShuffleSplit

# 1) قراءة الأسئلة | read the questions
qa_all = pd.DataFrame([json.loads(l) for l in open(DATA_DIR / 'bayan_qa_600.jsonl', encoding='utf-8')])
print('Questions:', len(qa_all), '| no answer:', int(qa_all.is_impossible.sum()))

# 2) تقسيم 85% تدريب و15% تحقق (النص نفسه لا يظهر في الطرفين) | split 85/15, same text never in both
a, b = next(GroupShuffleSplit(1, test_size=0.15, random_state=42).split(qa_all, groups=qa_all.context))
qa_train, qa_valid = qa_all.iloc[a].reset_index(drop=True), qa_all.iloc[b].reset_index(drop=True)

# 3) تحويل السؤال والنص إلى أرقام + موضع بداية ونهاية الإجابة | question + text -> numbers + start/end of the answer
def encode_qa(batch):
    enc = tokenizer(batch['question'], batch['context'], truncation='only_second',
                    max_length=256, return_offsets_mapping=True)
    starts, ends = [], []
    for i, offsets in enumerate(enc['offset_mapping']):
        answer = json.loads(batch['answers_text'][i])
        start_tok = end_tok = 0                          # 0 = لا إجابة | no answer
        if answer:
            s_char = json.loads(batch['answers_start'][i])[0]       # بداية الإجابة بالحروف | answer start (characters)
            e_char = s_char + len(answer[0])                        # نهايتها | answer end
            ctx = [j for j, sid in enumerate(enc.sequence_ids(i)) if sid == 1]   # رموز النص فقط | text tokens only
            start_tok = next((j for j in ctx if offsets[j][1] > s_char), 0)
            end_tok = [j for j in ctx if offsets[j][0] < e_char][-1]
        starts.append(start_tok); ends.append(end_tok)
    return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
            'start_positions': starts, 'end_positions': ends}

cols = ['question', 'context', 'answers_text', 'answers_start']
qa_train_ds = Dataset.from_pandas(qa_train[cols]).map(encode_qa, batched=True, remove_columns=cols)

# 4) التدريب | training
qa_args = TrainingArguments(output_dir='/content/tmp_qa', learning_rate=3e-5,
                            per_device_train_batch_size=16, num_train_epochs=1,
                            save_strategy='no', fp16=torch.cuda.is_available(), report_to='none')
qa_trainer = Trainer(model=AutoModelForQuestionAnswering.from_pretrained(CHECKPOINT),
                     args=qa_args, train_dataset=qa_train_ds,
                     data_collator=DataCollatorWithPadding(tokenizer))
qa_trainer.train()

# 5) الحفظ | save
qa_trainer.save_model('/content/bayan_qa_model')
tokenizer.save_pretrained('/content/bayan_qa_model')
print('✅ Saved / تم الحفظ: /content/bayan_qa_model')

Questions: 600 | no answer: 90


Map:   0%|          | 0/526 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
qa_outputs.weight           | MISSING    | 
qa_outputs.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved / تم الحفظ: /content/bayan_qa_model


## Step 6 — اختبار نموذج QA | Test the QA model
- 🇸🇦 نسأل النموذج أسئلة **لم يرها**. نستخدم الدالة `best_span` من Step 4 لنأخذ أفضل مقطع أو `None`.
- 🇬🇧 We ask questions the model **has not seen**. We reuse `best_span` from Step 4 to get the best span or `None`.

### Steps: QA Model Inference and Evaluation

1. **Load the Trained Model**
   - Load the fine-tuned Question Answering model and set it to evaluation mode.

2. **Prepare the Input**
   - Tokenize the question and context and generate offset mappings.
   - Keep only context token offsets.

3. **Generate Predictions**
   - Pass the inputs to the model to obtain Start and End Logits.

4. **Extract the Answer**
   - Use `best_span()` to select the best answer or return `None`.

5. **Evaluate on Validation Data**
   - Compare predicted answers with the Ground Truth answers.
   - Check whether unanswerable questions correctly return `None`.

6. **Print Evaluation Results**
   - Display the number of correct answers and correctly identified no-answer cases.

7. **Test Smoke Questions**
   - Run the model on 12 sample questions and display the predicted answers.

In [14]:
QA_CHECKPOINT = '/content/bayan_qa_model'   # النموذج الذي دربناه في الخطوة 5 | the model we trained in Step 5
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_CHECKPOINT)
qa_model.eval()

# سؤال + نص ← إجابة أو None | question + text -> answer or None
def answer(question, context):
    enc = tokenizer(question, context, truncation='only_second', max_length=256,
                    return_offsets_mapping=True, return_tensors='pt')
    kinds = enc.sequence_ids(0)                                    # 1 = رمز من النص | 1 = a text token
    offsets = enc.pop('offset_mapping')[0].tolist()
    offsets = [o if k == 1 else None for o, k in zip(offsets, kinds)]   # نتجاهل رموز السؤال | ignore question tokens
    with torch.no_grad():
        out = qa_model(**enc)
    return best_span(out.start_logits[0].numpy(), out.end_logits[0].numpy(), offsets, context)['answer']

# أسئلة التحقق (لم يرها النموذج) | validation questions (unseen)
ok_answer = ok_none = n_answer = n_none = 0
for r in qa_valid.itertuples():
    pred = answer(r.question, r.context)
    if r.is_impossible:                          # يجب أن يقول "لا إجابة" | should say "no answer"
        n_none += 1
        ok_none += (pred is None)
    else:                                        # يجب أن تطابق الإجابة الصحيحة | should match the gold answer
        n_answer += 1
        ok_answer += (pred or '').strip(' .') == json.loads(r.answers_text)[0].strip(' .')

print(f'Correct answers / إجابات صحيحة      : {ok_answer} / {n_answer}')
print(f'No-answer handled / "لا إجابة" صحيحة : {ok_none} / {n_none}')

# أسئلة الدخان الاثنا عشر | the 12 smoke questions
for q in qa_rows:
    print('-', q['question'], '->', answer(q['question'], q['context']))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Correct answers / إجابات صحيحة      : 5 / 68
No-answer handled / "لا إجابة" صحيحة : 0 / 6
- What is the refund processing fee? -> five business days
- Which browser is recommended? -> within two working days
- Can I use fingerprint login? -> three working days
- How long does a refund review take? -> five business days
- What should be saved when the portal shows an error? -> two working days
- How can users reset passwords? -> three working days
- How early should visitors arrive? -> 15 minutes before their appointment. Missing two appointments in a row blocks new bookings for 14 days. Reminders are sent by SMS 24 hours and 2 hours before the visit. Each booking covers one person
- How long does a building permit decision normally take? -> 15 working days
- Where can transport cards be topped up? -> within seven days of the trip. The last evening metro departs at 11:30 PM
- Who must remove construction debris? -> the same day
- When can customers request a water tanker? -> 48 hours in

## مقارنة النموذجين | The two models side by side

| | نموذج NER (Step 3) | نموذج QA (Step 5) |
|---|---|---|
| **المهمة \| Task** | استخراج الشخص والمكان والخدمة <br> Extract person, location, service | إيجاد جواب السؤال داخل نص، أو قول "لا إجابة" <br> Find the answer inside a text, or say "no answer" |
| **الملف \| Data file** | `bayan_ner_4000.csv` | `bayan_qa_600.jsonl` |
| **المدخل \| Input** | كلمات الجملة <br> Words of the sentence | السؤال والنص معًا <br> Question + text together |
| **المطلوب منه \| Output** | وسم لكل كلمة (B-/I-/O) <br> One tag per word | رقم رمز **بداية** الإجابة ورقم رمز **نهايتها** <br> Start and end token of the answer |
| **الرأس \| Model head** | `AutoModelForTokenClassification` | `AutoModelForQuestionAnswering` |
| **النموذج الأساسي \| Base model** | `xlm-roberta-base` | `xlm-roberta-base` |
| **التقسيم \| Split** | 85% تدريب، 15% تحقق <br> 85% train, 15% validation | 85% تدريب، 15% تحقق بحسب النص <br> 85/15, split by text |
| **المقياس \| Metric** | Entity-F1 (مكتبة `seqeval`) | الإجابات الصحيحة + عدد مرات قول "لا إجابة" بشكل صحيح <br> Correct answers + correct "no answer" |
| **المخرج \| Saved model** | `bayan_ner_model` | `bayan_qa_model` |

## أسئلة للنقاش | Think and discuss
1. لماذا تأخذ الأجزاء التابعة `-100`؟ / Why do continuation subwords get `-100`?
2. لماذا نسمح لنظام QA بإرجاع `None`؟ / Why let a QA system return `None`?
3. لماذا تصعب الكيانات العربية الملتصقة بحروف مثل "بالرياض"؟ / Why are Arabic clitic-attached locations (e.g. "بالرياض") hard?
